# 06_classical_feature_engineering

Publication-ready Colab notebook for the equine thermography Explainable AI study.


## Purpose
Extract classical image-derived features for baseline models. Features are computed from images only. Expert hotspot boxes/points are not used as predictors to avoid label leakage.

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, warnings, math, random
from datetime import datetime, timezone
import pandas as pd
import numpy as np

BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"
FEATURES_DIR = OUTPUT_ROOT / "features"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR,
          MODEL_SELECTION_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR, FEATURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("Project root:", PROJECT_ROOT)
print("Config dir:", CONFIG_DIR)

Project root: /content/project_thermography_equine
Config dir: /content/project_thermography_equine/outputs/config


In [2]:
required = [
    CONFIG_DIR / "analysis_config.json",
    CONFIG_DIR / "study_protocol.json",
    CONFIG_DIR / "master_metadata_qc.csv",
    CONFIG_DIR / "split_manifest.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 00-04 first. Missing:\n" + "\n".join(missing))

with open(CONFIG_DIR / "analysis_config.json", "r", encoding="utf-8") as f:
    analysis_config = json.load(f)
with open(CONFIG_DIR / "study_protocol.json", "r", encoding="utf-8") as f:
    study_protocol = json.load(f)

master = pd.read_csv(CONFIG_DIR / "master_metadata_qc.csv")
split_manifest = pd.read_csv(CONFIG_DIR / "split_manifest.csv")

required_cols = [
    "horse_id", "image_name", "split", "label_clinical", "label_binary",
    "relative_image_path", "included_in_final_analysis",
    "healthy_with_expert_hotspot", "annotation_label_conflict", "annotation_clinical_note"
]
missing_cols = [c for c in required_cols if c not in master.columns]
if missing_cols:
    raise KeyError("master_metadata_qc.csv is missing required columns from notebooks 02-04: " + ", ".join(missing_cols))

if master["annotation_label_conflict"].astype(bool).any():
    raise ValueError("True annotation-label conflicts are present. Resolve them before modeling.")

master = master[master["included_in_final_analysis"].astype(bool)].copy()
print("Included records:", len(master))
print("Healthy images with expert-marked hotspot retained:", int(master["healthy_with_expert_hotspot"].sum()))
display(master.groupby(["split", "label_clinical", "healthy_with_expert_hotspot"]).size().reset_index(name="n"))

Included records: 347
Healthy images with expert-marked hotspot retained: 6


,split,label_clinical,healthy_with_expert_hotspot,n
0,test,healthy,False,34
1,test,healthy,True,6
2,test,pathological,False,13
3,train,healthy,False,179
4,train,pathological,False,63
5,valid,healthy,False,38
6,valid,pathological,False,14


In [3]:


preprocess_path = CONFIG_DIR / "preprocess_manifest.csv"
if not preprocess_path.exists():
    raise FileNotFoundError(
        f"Missing {preprocess_path}. Run Notebook 05 first or upload the project outputs "
        "containing preprocess_manifest.csv."
    )

prep = pd.read_csv(preprocess_path)
required_prep_cols = ["split", "image_name", "clean_image_path"]
missing_prep_cols = [c for c in required_prep_cols if c not in prep.columns]
if missing_prep_cols:
    raise KeyError(
        "preprocess_manifest.csv is missing required columns: " + ", ".join(missing_prep_cols)
    )

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def is_image_file(path):
    return Path(path).suffix.lower() in IMAGE_EXTS

def count_images(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(1 for p in path.rglob("*") if p.is_file() and is_image_file(p))

def safe_extract_zip(zip_path, destination):
    """Safely extract a ZIP file without allowing path traversal."""
    zip_path = Path(zip_path)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    dest_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            member_path = destination / member.filename
            member_resolved = member_path.resolve()
            if not str(member_resolved).startswith(str(dest_resolved)):
                raise RuntimeError(f"Unsafe path detected inside ZIP: {member.filename}")
        zf.extractall(destination)
    return destination

def merge_tree(src, dst):
    """Merge files from src into dst while preserving relative paths."""
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    copied = 0
    for item in src.rglob("*"):
        if item.is_file():
            rel = item.relative_to(src)
            target = dst / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists() or target.stat().st_size != item.stat().st_size:
                shutil.copy2(item, target)
                copied += 1
    return copied

def find_zip_candidates(patterns, roots):
    candidates = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(root.glob(pattern))
            if root != BASE_DIR:
                candidates.extend(root.rglob(pattern))
    unique = []
    seen = set()
    for p in candidates:
        p = Path(p)
        if p.is_file() and p.suffix.lower() == ".zip" and p.resolve() not in seen:
            unique.append(p)
            seen.add(p.resolve())
    return sorted(unique, key=lambda p: p.stat().st_mtime, reverse=True)

def candidate_image_roots(extract_root, preferred_name=None):
    """Return extracted directories that contain images, prioritizing preferred folder names."""
    extract_root = Path(extract_root)
    roots = []
    if preferred_name:
        roots.extend([p for p in extract_root.rglob(preferred_name) if p.is_dir() and count_images(p) > 0])
    if count_images(extract_root) > 0:
        roots.append(extract_root)
    roots.extend([p for p in extract_root.rglob("*") if p.is_dir() and count_images(p) > 0])

    out = []
    seen = set()
    for p in roots:
        rp = p.resolve()
        if rp not in seen:
            out.append(p)
            seen.add(rp)
    return sorted(out, key=lambda p: count_images(p), reverse=True)


raw_zip_roots = [BASE_DIR, PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR]
raw_zip_patterns = ["dataset_split*.zip", "*dataset*split*.zip", "*raw*.zip"]
raw_zip_candidates = find_zip_candidates(raw_zip_patterns, raw_zip_roots)

if count_images(SPLIT_DATA_DIR) == 0 and raw_zip_candidates:
    print("Raw dataset ZIP candidates found:", [p.name for p in raw_zip_candidates[:5]])
    for zip_path in raw_zip_candidates:
        extract_root = DATA_ROOT / "_zip_extracts" / zip_path.stem
        print("Extracting raw ZIP:", zip_path)
        safe_extract_zip(zip_path, extract_root)
        roots = candidate_image_roots(extract_root, preferred_name="dataset_split")
        if roots:
            copied = merge_tree(roots[0], SPLIT_DATA_DIR)
            print("Merged raw images from", roots[0], "copied files:", copied)
            break
print("Raw images available in dataset_split:", count_images(SPLIT_DATA_DIR))


processed_zip_roots = [BASE_DIR, PROJECT_ROOT, DATA_ROOT, PROCESSED_DIR, RAW_DATA_DIR]
processed_zip_patterns = [
    "clean_images*.zip", "*clean*image*.zip", "*224x224*.zip",
    "*preprocess*.zip", "*processed*.zip"
]
processed_zip_candidates = find_zip_candidates(processed_zip_patterns, processed_zip_roots)

if count_images(CLEAN_IMAGE_DIR) == 0 and processed_zip_candidates:
    print("Preprocessed image ZIP candidates found:", [p.name for p in processed_zip_candidates[:10]])
    for zip_path in processed_zip_candidates:
        extract_root = PROCESSED_DIR / "_zip_extracts" / zip_path.stem
        print("Extracting preprocessed ZIP:", zip_path)
        safe_extract_zip(zip_path, extract_root)
        roots = candidate_image_roots(extract_root, preferred_name="clean_images")
        if not roots:
            print("No images found after extracting", zip_path)
            continue
        copied = merge_tree(roots[0], CLEAN_IMAGE_DIR)
        print("Merged clean images from", roots[0], "copied files:", copied)
        if count_images(CLEAN_IMAGE_DIR) > 0:
            break

print("Clean images available:", count_images(CLEAN_IMAGE_DIR))
if count_images(CLEAN_IMAGE_DIR) == 0:
    raise FileNotFoundError(
        "No preprocessed clean images were found. Upload clean_images_224x224.zip or rerun Notebook 05. "
        "Notebook 06 intentionally uses preprocessed images, not raw images, for classical feature extraction."
    )

print("preprocess_manifest rows:", len(prep))
print("Example clean_image_path values:")
display(prep[["split", "image_name", "clean_image_path"]].head())


if "clean_image_path" in master.columns:
    master = master.drop(columns=["clean_image_path"])
master = master.merge(
    prep[["split", "image_name", "clean_image_path"]].drop_duplicates(),
    on=["split", "image_name"],
    how="left"
)

if master["clean_image_path"].isna().any():
    missing_clean = master[master["clean_image_path"].isna()].copy()
    display(missing_clean[["split", "image_name"]].head(20))
    raise FileNotFoundError(
        f"{len(missing_clean)} rows did not receive clean_image_path from preprocess_manifest.csv."
    )


clean_image_files = [p for p in CLEAN_IMAGE_DIR.rglob("*") if p.is_file() and is_image_file(p)]
print("Clean image files indexed:", len(clean_image_files))

from collections import defaultdict
lookup_by_name = defaultdict(list)
for p in clean_image_files:
    lookup_by_name[p.name].append(p)

def path_candidates_from_saved_value(saved_path):
    saved = Path(str(saved_path))
    candidates = []
    if saved.is_absolute():
        candidates.append(saved)
    else:
        candidates.extend([PROJECT_ROOT / saved, DATA_ROOT / saved, BASE_DIR / saved, saved])

    parts = saved.parts

    for anchor, root in [
        ("project_thermography_equine", BASE_DIR / PROJECT_NAME),
        ("data", DATA_ROOT),
        ("processed", PROCESSED_DIR),
        ("clean_images", CLEAN_IMAGE_DIR),
    ]:
        if anchor in parts:
            idx = parts.index(anchor)
            tail = parts[idx + 1:]
            if anchor == "clean_images":
                candidates.append(root / Path(*tail))
            elif anchor == "processed" and len(tail) > 0 and tail[0] == "clean_images":
                candidates.append(PROCESSED_DIR / Path(*tail))
            elif anchor == "data" and len(tail) >= 2 and tail[0] == "processed" and tail[1] == "clean_images":
                candidates.append(DATA_ROOT / Path(*tail))
            elif anchor == "project_thermography_equine":
                candidates.append((BASE_DIR / PROJECT_NAME) / Path(*tail))
    return candidates

def resolve_clean_path(row):
    image_name = str(row["image_name"])
    split = str(row["split"])
    label = str(row["label_clinical"])
    folder_label = str(row["folder_label"]) if "folder_label" in row.index and pd.notna(row["folder_label"]) else label

    candidates = path_candidates_from_saved_value(row["clean_image_path"])
    candidates.extend([
        CLEAN_IMAGE_DIR / split / label / image_name,
        CLEAN_IMAGE_DIR / split / folder_label / image_name,
        CLEAN_IMAGE_DIR / label / image_name,
        CLEAN_IMAGE_DIR / folder_label / image_name,
        CLEAN_IMAGE_DIR / image_name,
    ])


    for p in lookup_by_name.get(image_name, []):
        p_parts = {part.lower() for part in p.parts}
        score = int(split.lower() in p_parts) + int(label.lower() in p_parts) + int(folder_label.lower() in p_parts)
        candidates.append((score, p))

    scored = []
    for c in candidates:
        if isinstance(c, tuple):
            scored.append(c)
        else:
            scored.append((0, Path(c)))
    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    for _, c in scored:
        c = Path(c)
        if c.exists() and c.is_file():
            return str(c)
    return ""

master["clean_image_path"] = master.apply(resolve_clean_path, axis=1)
image_path_col = "clean_image_path"
missing = master[image_path_col].eq("") | ~master[image_path_col].apply(lambda x: Path(str(x)).exists())

if missing.any():
    cols = [c for c in ["split", "label_clinical", "folder_label", "image_name", "clean_image_path"] if c in master.columns]
    display(master.loc[missing, cols].head(30))
    raise FileNotFoundError(
        f"{int(missing.sum())} clean image paths are missing for feature engineering. "
        "Check that clean_images_224x224.zip corresponds to the same dataset used in notebooks 00-05."
    )

print("Using image path column:", image_path_col)
print("Images available:", int((~missing).sum()), "/", len(master))
display(master.groupby(["split", "label_clinical"]).size().reset_index(name="n"))


Raw dataset ZIP candidates found: ['dataset_split-20260605T090743Z-3-001.zip']
Extracting raw ZIP: /content/dataset_split-20260605T090743Z-3-001.zip
Merged raw images from /content/project_thermography_equine/data/_zip_extracts/dataset_split-20260605T090743Z-3-001/dataset_split copied files: 347
Raw images available in dataset_split: 347
Preprocessed image ZIP candidates found: ['clean_images_224x224.zip']
Extracting preprocessed ZIP: /content/clean_images_224x224.zip
Merged clean images from /content/project_thermography_equine/data/processed/_zip_extracts/clean_images_224x224/clean_images copied files: 347
Clean images available: 347
preprocess_manifest rows: 347
Example clean_image_path values:


,split,image_name,clean_image_path
0,test,0A0F5.jpg,/content/project_thermography_equine/data/proc...
1,test,1CGFM.jpg,/content/project_thermography_equine/data/proc...
2,test,1RDFC.jpg,/content/project_thermography_equine/data/proc...
3,test,31W0A.jpg,/content/project_thermography_equine/data/proc...
4,test,6BGWZ.jpg,/content/project_thermography_equine/data/proc...


Clean image files indexed: 347
Using image path column: clean_image_path
Images available: 347 / 347


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


In [4]:
from PIL import Image
try:
    from scipy.stats import entropy as scipy_entropy
except Exception:
    scipy_entropy = None

def image_entropy(gray):
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 255), density=True)
    hist = hist[hist > 0]
    return float(-(hist * np.log2(hist)).sum())

def gradient_features(gray):
    gy, gx = np.gradient(gray.astype(np.float32))
    mag = np.sqrt(gx * gx + gy * gy)
    return {
        "grad_mean": float(np.mean(mag)),
        "grad_std": float(np.std(mag)),
        "grad_p90": float(np.percentile(mag, 90)),
        "grad_p95": float(np.percentile(mag, 95)),
    }

def extract_features(path):
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img).astype(np.float32)
    gray = np.asarray(img.convert("L")).astype(np.float32)
    h, w = gray.shape
    left = gray[:, :w//2]
    right = np.fliplr(gray[:, w - w//2:])
    min_w = min(left.shape[1], right.shape[1])
    asym = left[:, :min_w] - right[:, :min_w]
    vals = gray.ravel()
    top10_thr = np.percentile(vals, 90)
    top30_thr = np.percentile(vals, 70)
    top10 = vals[vals >= top10_thr]
    top30 = vals[vals >= top30_thr]
    feats = {
        "img_mean": float(np.mean(vals)),
        "img_std": float(np.std(vals)),
        "img_min": float(np.min(vals)),
        "img_max": float(np.max(vals)),
        "img_p05": float(np.percentile(vals, 5)),
        "img_p10": float(np.percentile(vals, 10)),
        "img_p50": float(np.percentile(vals, 50)),
        "img_p90": float(np.percentile(vals, 90)),
        "img_p95": float(np.percentile(vals, 95)),
        "img_p99": float(np.percentile(vals, 99)),
        "top10_mean": float(np.mean(top10)),
        "top10_std": float(np.std(top10)),
        "top10_area_fraction": float(len(top10) / len(vals)),
        "top30_mean": float(np.mean(top30)),
        "top30_std": float(np.std(top30)),
        "top30_area_fraction": float(len(top30) / len(vals)),
        "entropy_gray": image_entropy(gray),
        "lr_asym_mean_abs": float(np.mean(np.abs(asym))),
        "lr_asym_max_abs": float(np.max(np.abs(asym))),
        "lr_asym_std": float(np.std(asym)),
        "height_px_features": int(h),
        "width_px_features": int(w),
    }
    # Channel-level pseudocolor summaries are included as image-derived features.
    for i, ch in enumerate(["r", "g", "b"]):
        c = arr[:, :, i].ravel()
        feats[f"{ch}_mean"] = float(np.mean(c))
        feats[f"{ch}_std"] = float(np.std(c))
        feats[f"{ch}_p90"] = float(np.percentile(c, 90))
    feats.update(gradient_features(gray))
    return feats

rows = []
for _, row in master.iterrows():
    try:
        feats = extract_features(row[image_path_col])
    except Exception as e:
        raise RuntimeError(f"Feature extraction failed for {row.get('image_name', 'unknown image')} at {row[image_path_col]}: {e}") from e
    base_cols = [
        "horse_id", "image_name", "split", "label_clinical", "label_binary",
        "relative_image_path", "healthy_with_expert_hotspot", "annotation_label_conflict",
        "annotation_clinical_note", "included_in_final_analysis"
    ]
    out = {c: row[c] for c in base_cols if c in row.index}
    out["feature_image_path"] = str(row[image_path_col])
    out.update(feats)
    rows.append(out)

features = pd.DataFrame(rows)
print("Feature table:", features.shape)
display(features.groupby(["split", "label_clinical"]).size().reset_index(name="n"))

Feature table: (347, 46)


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


In [5]:
# Define model feature columns and save outputs.
non_feature_cols = {
    "horse_id", "image_name", "split", "label_clinical", "label_binary",
    "relative_image_path", "feature_image_path", "healthy_with_expert_hotspot",
    "annotation_label_conflict", "annotation_clinical_note", "included_in_final_analysis"
}
feature_columns = [c for c in features.columns if c not in non_feature_cols and pd.api.types.is_numeric_dtype(features[c])]
if not feature_columns:
    raise RuntimeError("No numeric feature columns were generated.")

features[feature_columns] = features[feature_columns].replace([np.inf, -np.inf], np.nan)
if features[feature_columns].isna().any().any():
    features[feature_columns] = features[feature_columns].fillna(features[feature_columns].median(numeric_only=True))

features.to_csv(CONFIG_DIR / "classical_features.csv", index=False)
features.to_csv(FEATURES_DIR / "classical_features.csv", index=False)
features.to_csv(REPORTS_DIR / "classical_features.csv", index=False)

feature_metadata = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "n_images": int(len(features)),
    "n_features": int(len(feature_columns)),
    "feature_columns": feature_columns,
    "leakage_policy": "Expert hotspot annotations are retained as metadata but excluded from predictors.",
    "clinical_label_source": "expert_classification_from_folder_structure"
}
(CONFIG_DIR / "classical_feature_metadata.json").write_text(json.dumps(feature_metadata, indent=2), encoding="utf-8")
(REPORTS_DIR / "classical_feature_metadata.json").write_text(json.dumps(feature_metadata, indent=2), encoding="utf-8")

feature_summary = pd.DataFrame([{
    "n_images": len(features),
    "n_feature_columns": len(feature_columns),
    "healthy_with_expert_hotspot_retained": int(features["healthy_with_expert_hotspot"].sum())
}])
feature_summary.to_csv(CONFIG_DIR / "classical_feature_summary.csv", index=False)
feature_summary.to_csv(REPORTS_DIR / "classical_feature_summary.csv", index=False)
print("Saved classical features:", len(feature_columns), "predictors")

Saved classical features: 35 predictors
